# UdaPlay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Optional SQLite compatibility for the Udacity workspace.
# pysqlite3 is imported dynamically so local editors do not warn when it is absent.

from importlib import import_module, util
import sys

if util.find_spec("pysqlite3") is not None:
    pysqlite3 = import_module("pysqlite3")
    sys.modules["sqlite3"] = pysqlite3

In [12]:
import os

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.evaluation import AgentEvaluator, TestCase
from lib.llm import LLM
from lib.messages import AIMessage, SystemMessage, ToolMessage, UserMessage
from lib.parsers import PydanticOutputParser
from lib.tooling import tool
from lib.vector_db import VectorStore


In [13]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

assert OPENAI_API_KEY is not None, "OPENAI_API_KEY must be set in your local .env file"
assert TAVILY_API_KEY is not None, "TAVILY_API_KEY must be set in your local .env file"

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [14]:
# Connect to the collection once, reusing the same embedding function as Part 01
# so query text is embedded into the same vector space as the stored games.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
)
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)


@tool
def retrieve_game(query: str):
    """Semantic search: Finds most similar results in the vector DB

    args:
    - query: a question about game industry. 
    """

    # Embed the query and find the nearest stored games
    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    # Chroma returns a list per query; we only sent one query
    metadatas = results["metadatas"][0]
    documents = results["documents"][0]

    # Build a clean list to return
    formatted_results = []
    for metadata, document in zip(metadatas, documents):
        formatted_results.append({
            "Platform": metadata.get("Platform", "Unknown"),
            "Name": metadata.get("Name", "Unknown"),
            "YearOfRelease": metadata.get("YearOfRelease", "Unknown"),
            "Description": document
        })

    return formatted_results

#### Evaluate Retrieval Tool

In [10]:
class EvaluationReport(BaseModel):
    useful: bool = Field(
        description="Whether the retrieved documents are enough to answer the question"
    )
    description: str = Field(
        description="Detailed explanation of the evaluation result"
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> EvaluationReport:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    """

    # Ask an LLM to judge whether the documents can answer the question
    llm = LLM(model="gpt-4o-mini", temperature=0.0)

    system_message = SystemMessage(
        content=(
            "Your task is to evaluate if the documents are enough to respond the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not."
        )
    )
    user_message = UserMessage(
        content=f"Question: {question}\nRetrieved documents: {retrieved_docs}"
    )

    # Request a structured response and parse it into EvaluationReport
    ai_message = llm.invoke(
        input=[system_message, user_message],
        response_format=EvaluationReport,
    )
    parser = PydanticOutputParser(model_class=EvaluationReport)
    return parser.parse(ai_message)

In [19]:
# Quick check: a relevant query should be useful, an off-topic one should not.
relevant_question = "What is Mario Kart?"
relevant_docs = retrieve_game(relevant_question)
relevant_report = evaluate_retrieval(relevant_question, relevant_docs)
print("useful:", relevant_report.useful)
print("description:", relevant_report.description)

print("-" * 40)

offtopic_question = "What is the best recipe for pizza?"
offtopic_docs = retrieve_game(offtopic_question)
offtopic_report = evaluate_retrieval(offtopic_question, offtopic_docs)
print("useful:", offtopic_report.useful)
print("description:", offtopic_report.description)

useful: True
description: The retrieved documents provide sufficient information to answer the question "What is Mario Kart?". Specifically, two documents focus on the Mario Kart series: one describes 'Mario Kart 8 Deluxe', which is an enhanced version of 'Mario Kart 8' released in 2017 for the Nintendo Switch, and the other describes 'Super Mario Kart', the original game in the series released in 1992 for the Super Nintendo Entertainment System. Both documents detail the gameplay mechanics, the context of the games within the Mario universe, and their significance in the kart racing genre. 

While there are additional documents about other racing games like 'Gran Turismo', they are not relevant to the question about Mario Kart. The key documents provide a clear understanding of what Mario Kart is, including its gameplay style, character involvement, and its historical importance in gaming. Therefore, the documents are indeed enough to respond to the query.
----------------------------

#### Game Web Search Tool

In [22]:
# Create the Tavily client once and reuse it for every web search
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def game_web_search(question: str):
    """Search the web for information about the game industry.

    args:
    - question: a question about game industry.
    """

    # advanced search depth gives deeper retrieval; include_answer adds a synthesized answer
    search_results = tavily_client.search(
        question,
        search_depth="advanced",
        max_results=5,
        include_answer=True,
    )

    # Keep only the parts useful to the agent
    results = [
        {
            "title": result["title"],
            "url": result["url"],
            "content": result["content"],
        }
        for result in search_results.get("results", [])
    ]

    return {
        "answer": search_results.get("answer"),
        "results": results,
    }


### Agent

In [23]:
agent = Agent(
    model_name="gpt-4o-mini",
    temperature=0.0,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=(
        "You are UdaPlay, a helpful research assistant for the video game industry. "
        "First use retrieve_game to search the internal vector database, then use "
        "evaluate_retrieval to check whether the results can answer the question. "
        "If they are not enough, use game_web_search to look the answer up on the web. "
        "Base your answers on the information returned by the tools and cite the game "
        "details (name, platform, year) when relevant."
    ),
)


In [24]:
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for question in questions:
    run = agent.invoke(question)
    final_state = run.get_final_state()
    answer = final_state["messages"][-1].content
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 60)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: When was Pokémon Gold and Silver released?
A: Pokémon Gold and Silver were released in 1999 for the Game Boy Color.
------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: Which one was the first 3D platformer Mario game?
A: The first 3D platformer Mario game is **Super Mario 64**, which was 

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes